In [ ]:
# import libraries
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 


In [ ]:
# lets import the dataset (here we are gonna use processed_datatcsv)
df = pd.read_csv("processed_data.csv")

In [ ]:
df


In [ ]:
# lets drop the columns
df.drop(columns = "Unnamed: 0",inplace = True)

In [ ]:
df.sample(5)


In [ ]:
# here we train different Ml models and  
# we work with different models


In [ ]:
# here we do not use the the straify  = labels 


# lets now do word to number transfromation
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest = train_test_split(df.drop("status", axis=1),df["status"],test_size = 0.2,random_state=1,stratify=df["status"])

In [ ]:
xtrain.shape

In [ ]:
xtest.shape

In [ ]:
ytrain.shape

In [ ]:
ytest.shape

In [ ]:
# now lets make a pipeline for testing the logistic regression model using tf-idf vectorizer


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

num_cols = ["statement_length", "num_words", "vocabulary_size", "avg_word_length"]

preprocessor = ColumnTransformer([
    ("text", TfidfVectorizer(), "clean_text"),   # preprocessed text
    ("num", "passthrough", num_cols)             # numeric features
])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
param_grid = {
    # TF-IDF tuning
    "preprocess__text__max_features": [5000, 10000, 15000],
    "preprocess__text__ngram_range": [(1,1), (1,2)],
    
    
    # Logistic Regression tuning
    "model__C": [0.1, 1, 10],
}

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=3
)

grid.fit(xtrain, ytrain)

In [ ]:
# now lets call our everymodels for checking there base performance
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


# Ensemble models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier

# Advanced boosting
from sklearn.neighbors import KNeighborsClassifier

# External libraries
# from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier
# from catboost import CatBoostClassifier


In [ ]:
models = {
    "MultinomialNB": MultinomialNB(),
    "BernoulliNB": BernoulliNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(),
     

    "KNN": KNeighborsClassifier(),

    "RandomForest": RandomForestClassifier(),
    "Bagging": BaggingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "GradientBoosting": GradientBoostingClassifier(),

    # "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
    # "LightGBM": LGBMClassifier(),
    # "CatBoost": CatBoostClassifier(verbose=0)
}

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,roc_curve, auc)

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score
)
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

results = []

for name, model in models.items():
    print(f"\n===== {name} =====")
    
    # Train
    model.fit(xtrain_final, ytrain)
    
    # Predict
    y_pred = model.predict(xtest_final)
    
    # Accuracy
    acc = accuracy_score(ytest, y_pred)
    
    # Classification report
    report = classification_report(ytest, y_pred, output_dict=True)
    
    precision = report["weighted avg"]["precision"]
    recall = report["weighted avg"]["recall"]
    f1 = report["weighted avg"]["f1-score"]
    
    # Store results
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })
    
    # Print report
    print(classification_report(ytest, y_pred))
    
    # Confusion matrix
    cm = confusion_matrix(ytest, y_pred)
    
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues")
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
results